# 4장 모수적 생존모형과 외삽 (R로 하는 연구 분석)

보건통계학 기초 23장 d 절의 가상 시험(표준요법군 300명)의 전체생존 자료로 여섯 분포와 스플라인 모형을 맞추고 20년까지 외삽합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요.


## a. 자료와 관찰한 생존곡선

### 셀 1. 자료 불러오기

In [ ]:
install.packages(c("survival", "flexsurv"))   # 처음 한 번만
library(survival)
library(flexsurv)

options(HTTPUserAgent = "Mozilla/5.0")   # 사이트가 프로그램의 기본 요청을 막을 때를 대비해 브라우저처럼 요청
BASE <- "https://socialp-ajou.tecentriq12.workers.dev/data/"
d <- read.csv(paste0(BASE, "rcc_trial.csv"))
str(d)
table(d$event)

### 셀 2. 관찰한 생존곡선(Kaplan-Meier)

In [ ]:
km <- survfit(Surv(months, event) ~ 1, data = d)
km
summary(km, times = c(6, 12, 24))

## b. 곡선 하나 맞추기

### 셀 3. 와이블 곡선 맞추기

In [ ]:
fit_w <- flexsurvreg(Surv(months, event) ~ 1, data = d, dist = "weibull")
fit_w

### 셀 4. 정한 시점의 생존율

In [ ]:
set.seed(2026)   # 신뢰구간을 모의실험으로 구하므로 고정
s_w <- summary(fit_w, t = c(24, 60, 120, 240), type = "survival", tidy = TRUE)
round(s_w, 4)

## c. 여섯 분포 비교

### 셀 5. 여섯 분포를 맞추고 AIC·BIC 비교

In [ ]:
dists <- c(exp = "exp", weibull = "weibull", gompertz = "gompertz",
           lnorm = "lnorm", llogis = "llogis", gengamma = "gengamma")
fits <- lapply(dists, function(k)
  flexsurvreg(Surv(months, event) ~ 1, data = d, dist = k))

fit_tab <- data.frame(
  model = names(fits),
  k     = sapply(fits, function(f) f$npars),
  AIC   = sapply(fits, AIC),
  BIC   = sapply(fits, BIC))
fit_tab$dAIC <- fit_tab$AIC - min(fit_tab$AIC)
fit_tab[order(fit_tab$AIC), ]

## d. 20년까지의 생존율과 평균 생존

### 셀 6. 2·5·10·20년 생존율

In [ ]:
times <- c(24, 60, 120, 240)            # 2, 5, 10, 20년 (개월)
surv_at <- function(f) summary(f, t = times, type = "survival",
                               ci = FALSE, tidy = TRUE)$est
ext <- t(sapply(fits, surv_at))
colnames(ext) <- paste0(times / 12, "y")
round(100 * ext, 1)

### 셀 7. 20년까지의 평균 생존과 제한 없는 평균 (년)

In [ ]:
rmst20 <- sapply(fits, function(f)
  summary(f, type = "rmst", t = 240, ci = FALSE, tidy = TRUE)$est)
mean_all <- sapply(fits, function(f)
  summary(f, type = "mean", ci = FALSE, tidy = TRUE)$est)
round(data.frame(rmst_20y = rmst20 / 12, mean_unrestricted = mean_all / 12), 2)

## e. 그림으로 점검

### 셀 8. 관찰 구간과 외삽 구간의 곡선

In [ ]:
library(ggplot2)
grid <- seq(0, 240, by = 1)
curves <- do.call(rbind, lapply(names(fits), function(m)
  data.frame(model = m, months = grid,
             surv = summary(fits[[m]], t = grid, ci = FALSE, tidy = TRUE)$est)))
km_df <- data.frame(months = c(0, km$time), surv = c(1, km$surv))

ggplot(curves, aes(months / 12, surv, colour = model)) +
  geom_line(linewidth = 0.8) +
  geom_step(data = km_df, aes(months / 12, surv), colour = "black",
            inherit.aes = FALSE) +
  geom_vline(xintercept = 30 / 12, linetype = "dashed") +
  labs(x = "Years since randomisation", y = "Overall survival", colour = NULL) +
  theme_minimal(base_size = 13)

### 셀 9. 위험률: 자료에서 본 것과 모형이 그리는 것

In [ ]:
library(muhaz)
emp <- muhaz(d$months, d$event, max.time = 28)      # 자료에서 본 위험률(평활)
haz <- do.call(rbind, lapply(names(fits), function(m)
  data.frame(model = m, months = grid[grid > 0],
             h = summary(fits[[m]], t = grid[grid > 0], type = "hazard",
                         ci = FALSE, tidy = TRUE)$est)))

ggplot(haz, aes(months / 12, h * 12, colour = model)) +
  geom_line(linewidth = 0.8) +
  geom_line(data = data.frame(months = emp$est.grid, h = emp$haz.est),
            aes(months / 12, h * 12), colour = "black", linewidth = 1.1,
            inherit.aes = FALSE) +
  coord_cartesian(ylim = c(0, 1.2)) +
  labs(x = "Years since randomisation", y = "Hazard (deaths per person-year)",
       colour = NULL) +
  theme_minimal(base_size = 13)

## f. 스플라인과 모수 표현

### 셀 10. 스플라인 모형(매듭 1개, 2개)

In [ ]:
spl <- lapply(c(k1 = 1, k2 = 2), function(k)
  flexsurvspline(Surv(months, event) ~ 1, data = d, k = k, scale = "hazard"))
data.frame(
  model = names(spl),
  AIC   = sapply(spl, AIC),
  surv_10y = sapply(spl, function(f)
    summary(f, t = 120, ci = FALSE, tidy = TRUE)$est),
  rmst_20y = sapply(spl, function(f)
    summary(f, type = "rmst", t = 240, ci = FALSE, tidy = TRUE)$est) / 12)

### 셀 11. 같은 와이블의 두 가지 모수 표현

In [ ]:
fit_wph <- flexsurvreg(Surv(months, event) ~ 1, data = d, dist = "weibullPH")
rbind(AFT = fit_w$res[, "est"], PH = fit_wph$res[, "est"])
c(AFT = AIC(fit_w), PH = AIC(fit_wph))

## g. 과제 정답

### 셀 12. 과제 1 정답: 시간 단위를 년으로

In [ ]:
d$years <- d$months / 12
fit_wy <- flexsurvreg(Surv(years, event) ~ 1, data = d, dist = "weibull")
fit_wy$res[, "est"]
c(months = AIC(fit_w), years = AIC(fit_wy), diff = AIC(fit_w) - AIC(fit_wy))
summary(fit_wy, t = 20, ci = FALSE, tidy = TRUE)$est

### 셀 13. 과제 2 정답: 18개월에 마감한 자료로 다시

In [ ]:
d18 <- d
d18$event  <- ifelse(d$months > 18, 0, d$event)   # 18개월에 자료를 마감했다면
d18$months <- pmin(d$months, 18)
fits18 <- lapply(dists, function(k)
  flexsurvreg(Surv(months, event) ~ 1, data = d18, dist = k))
data.frame(
  events = sum(d18$event),
  AIC    = round(sapply(fits18, AIC), 1),
  surv_10y_18m = round(100 * sapply(fits18, function(f)
    summary(f, t = 120, ci = FALSE, tidy = TRUE)$est), 1),
  surv_10y_30m = round(100 * ext[, "10y"], 1))

### 셀 14. 과제 3 정답: 외부 자료와 비교

In [ ]:
ext5 <- 100 * ext[, "5y"]
round(sort(abs(ext5 - 20)), 1)     # 등록자료의 5년 생존율 20%(가상)와의 차이